# MVPA L2 Result Visualization

This notebook visualizes the MVPA L2 analysis outputs described in `PROJECT_CONTEXT.md`. It is a new additive notebook: it discovers and reads existing output tables/checkpoints, writes figures to a visualization folder, and does not modify any analysis scripts.

The notebook follows the planned hierarchy: Aim 1 decoding and specificity, Aim 2 SAD-HC neural differences, Aim 3 clinical relevance, Aim 4 neural-SCR convergence, Aim 5 oxytocin modulation, and sensitivity analyses.


## Interpretation Guardrails

- Decoding accuracy shows condition-relevant information is present; geometry, certainty, trajectories, clinical scores, SCR, and drug effects carry interpretation.
- Placebo diagnostic analyses and oxytocin-modulation analyses are kept separate.
- Primary neural metrics and primary SCR/clinical endpoints are shown before secondary or sensitivity results.
- Missing tables are reported explicitly rather than silently skipped.


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import joblib
except Exception:
    joblib = None

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False})

PRIMARY_NEURAL_METRICS = [
    "Neural_Dist_Safety_Background",
    "Neural_Dist_Threat_Safety",
    "Neural_Decision_Margin_CSS",
    "Neural_Decision_Margin_CSR",
    "Neural_Safety_Trajectory_Slope",
    "Neural_Threat_Trajectory_Slope",
]
SECONDARY_NEURAL_METRICS = ["Neural_Dist_Threat_Background", "Neural_Boundary_Separation", "Neural_Decoder_Entropy_CSS", "Neural_Decoder_Entropy_CSR"]
PRIMARY_CLINICAL_SCORES = ["lsas_total", "dass_anxiety"]
CLINICAL_SCORE_HIERARCHY = ["lsas_total", "lsas_fear", "lsas_avoid", "dass_anxiety"]
PRIMARY_SCR_INDICES = ["SCR_Safety_Trajectory_Slope", "SCR_Threat_Trajectory_Slope"]
SECONDARY_SCR_INDICES = ["SCR_SafetyMinusBackground", "SCR_ThreatMinusSafety"]
SCR_SENSITIVITY_FLAGS = ["SCR_Physiological_Responder", "SCR_Simple_Acquisition_Differential_Learner", "SCR_Habituation_Adjusted_Learner", "SCR_Late_Phase_Sensitivity_Learner"]
AIM_TABLES = {
    "Aim 2 group difference": "aim2_group_difference.csv",
    "Aim 3 clinical relevance": "aim3_clinical_relevance.csv",
    "Aim 4 SCR convergence": "aim4_scr_convergence.csv",
    "Aim 5 oxytocin modulation": "aim5_oxytocin_modulation.csv",
    "Sensitivity models": "sensitivity_models_all.csv",
    "Manuscript primary results": "manuscript_primary_results.csv",
    "Aim 4 convergence matrix": "aim4_convergence_matrix.csv",
    "Aim 1 SCR sensitivity": "aim1_scr_sensitivity.csv",
}
PALETTE = {"SAD": "#B5525C", "HC": "#2F6F73", "Placebo": "#5B7C99", "Oxytocin": "#C47D2F"}


In [ ]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "PROJECT_CONTEXT.md").exists():
            return path
    return start

REPO_ROOT = find_repo_root()
RESULT_ROOT_CANDIDATES = [REPO_ROOT / "results" / "outputs" / "mvpa_l2", REPO_ROOT / "outputs" / "mvpa_l2", REPO_ROOT / "code" / "outputs" / "mvpa_l2"]
RESULT_ROOT = next((path for path in RESULT_ROOT_CANDIDATES if path.exists()), RESULT_ROOT_CANDIDATES[0])
STATS_DIR = next((path for path in [RESULT_ROOT / "stats", REPO_ROOT / "results" / "mvpa_l2" / "stats"] if path.exists()), RESULT_ROOT / "stats")
HARMONIZED_DIR = RESULT_ROOT / "harmonized"
FIGURE_DIR = RESULT_ROOT / "figures" / "visualization_mvpa_l2"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"MVPA L2 result root: {RESULT_ROOT}")
print(f"Stats directory: {STATS_DIR}")
print(f"Notebook figure output: {FIGURE_DIR}")


In [ ]:
def read_csv_if_exists(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Could not read {path}: {exc}")
        return pd.DataFrame()

def savefig(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved figure: {path}")

def available_columns(df, cols):
    return [col for col in cols if col in df.columns]

def display_table(df, n=20, caption=None):
    if caption:
        display(Markdown(f"**{caption}**"))
    if df.empty:
        display(Markdown("_No rows available._"))
    else:
        display(df.head(n))

def coerce_numeric(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

def add_metric_role(df):
    out = df.copy()
    if "metric" in out.columns and "metric_role" not in out.columns:
        out["metric_role"] = np.where(out["metric"].isin(PRIMARY_NEURAL_METRICS), "primary", np.where(out["metric"].isin(SECONDARY_NEURAL_METRICS), "secondary", "other"))
    return out

def clean_label(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_") or "value"


## Output Inventory

The notebook first checks which documented post-Hyak tables and harmonized files are present. If the Aim 2-5 tables are missing, run the post-Hyak workflow and re-run this notebook.


In [ ]:
inventory_rows = []
for label, filename in AIM_TABLES.items():
    path = STATS_DIR / filename
    inventory_rows.append({"family": "stats", "label": label, "path": str(path.relative_to(REPO_ROOT)) if path.exists() else str(path), "exists": path.exists()})
for filename in ["mvpa_l2_subject_metrics.csv", "scr_sensitivity_groups.csv"]:
    path = HARMONIZED_DIR / filename
    inventory_rows.append({"family": "harmonized", "label": filename, "path": str(path.relative_to(REPO_ROOT)) if path.exists() else str(path), "exists": path.exists()})
for path in sorted(RESULT_ROOT.glob("**/*.joblib"))[:200]:
    inventory_rows.append({"family": "joblib", "label": path.name, "path": str(path.relative_to(REPO_ROOT)), "exists": True})
inventory = pd.DataFrame(inventory_rows)
display_table(inventory.sort_values(["family", "label"]), n=120, caption="Discovered MVPA L2 outputs")


In [ ]:
subject_metrics = read_csv_if_exists(HARMONIZED_DIR / "mvpa_l2_subject_metrics.csv")
scr_groups = read_csv_if_exists(HARMONIZED_DIR / "scr_sensitivity_groups.csv")
stats_tables = {label: add_metric_role(read_csv_if_exists(STATS_DIR / filename)) for label, filename in AIM_TABLES.items()}

print(f"Subject metrics rows: {len(subject_metrics)}")
print(f"SCR sensitivity rows: {len(scr_groups)}")
for label, table in stats_tables.items():
    print(f"{label}: {len(table)} rows")


## Cohort, Missingness, And SCR Sensitivity Checks

These descriptive checks help verify sample composition and responder/learner sensitivity cohorts before reading inferential models.


In [ ]:
if subject_metrics.empty:
    display(Markdown("_No harmonized `mvpa_l2_subject_metrics.csv` table is available yet._"))
else:
    count_cols = available_columns(subject_metrics, ["FeatureSpace", "Group", "Drug"])
    if count_cols:
        display_table(subject_metrics.groupby(count_cols, dropna=False).size().reset_index(name="n_rows"), n=100, caption="Rows by feature space, group, and drug")
    metric_cols = available_columns(subject_metrics, PRIMARY_NEURAL_METRICS + SECONDARY_NEURAL_METRICS + CLINICAL_SCORE_HIERARCHY + PRIMARY_SCR_INDICES + SECONDARY_SCR_INDICES)
    if metric_cols:
        missing = subject_metrics[metric_cols].isna().mean().sort_values(ascending=False).reset_index()
        missing.columns = ["variable", "missing_fraction"]
        plt.figure(figsize=(8, max(3, 0.25 * len(missing))))
        sns.barplot(data=missing, y="variable", x="missing_fraction", color="#5B7C99")
        plt.xlabel("Missing fraction")
        plt.ylabel("")
        plt.title("Subject-level missingness")
        savefig("subject_metric_missingness.png")
        plt.show()

if scr_groups.empty:
    display(Markdown("_No `scr_sensitivity_groups.csv` table is available._"))
else:
    display_table(scr_groups.head(20), n=20, caption="SCR sensitivity table preview")
    flag_cols = available_columns(scr_groups, SCR_SENSITIVITY_FLAGS)
    group_cols = available_columns(scr_groups, ["Group", "Drug"])
    if flag_cols:
        long_flags = scr_groups.melt(id_vars=group_cols, value_vars=flag_cols, var_name="cohort", value_name="included")
        long_flags["included"] = long_flags["included"].astype(bool)
        if group_cols:
            cohort_counts = long_flags.groupby(group_cols + ["cohort"], dropna=False)["included"].sum().reset_index(name="n_included")
            plt.figure(figsize=(10, max(4, 0.35 * len(flag_cols))))
            sns.barplot(data=cohort_counts, y="cohort", x="n_included", hue=group_cols[0], palette=PALETTE)
            plt.xlabel("Included subjects")
            plt.ylabel("")
            plt.title("SCR sensitivity cohort sizes")
            savefig("scr_sensitivity_cohort_sizes.png")
            plt.show()
        else:
            display_table(long_flags.groupby("cohort")["included"].sum().reset_index(name="n_included"), n=20, caption="SCR sensitivity cohort sizes")


## Subject-Level Primary Neural Metrics

When `mvpa_l2_subject_metrics.csv` exists, this plot shows descriptive distributions for the primary neural metrics by group and drug.


In [ ]:
if subject_metrics.empty:
    display(Markdown("_No subject-level neural metrics to plot yet._"))
else:
    metric_cols = available_columns(subject_metrics, PRIMARY_NEURAL_METRICS)
    id_cols = available_columns(subject_metrics, ["sub_ID", "FeatureSpace", "Group", "Drug"])
    if not metric_cols or "Group" not in subject_metrics.columns:
        display(Markdown("_The subject metrics table does not include enough primary metric/group columns for this plot._"))
    else:
        plot_df = subject_metrics[id_cols + metric_cols].melt(id_vars=id_cols, value_vars=metric_cols, var_name="metric", value_name="value")
        plot_df["value"] = pd.to_numeric(plot_df["value"], errors="coerce")
        plot_df = plot_df.dropna(subset=["value"])
        if "FeatureSpace" in plot_df.columns and plot_df["FeatureSpace"].notna().any():
            feature_space = "FearNetwork" if "FearNetwork" in set(plot_df["FeatureSpace"].dropna()) else plot_df["FeatureSpace"].dropna().iloc[0]
            plot_df = plot_df[plot_df["FeatureSpace"].eq(feature_space)]
        else:
            feature_space = "available feature space"
        if not plot_df.empty:
            g = sns.catplot(data=plot_df, x="Group", y="value", hue="Drug" if "Drug" in plot_df.columns else None, col="metric", col_wrap=3, kind="box", sharey=False, height=3.2, aspect=1.1, palette=PALETTE)
            g.set_titles("{col_name}")
            g.set_axis_labels("", "Metric value")
            g.fig.suptitle(f"Primary neural metrics: {feature_space}", y=1.03)
            savefig("primary_neural_metrics_by_group_drug.png")
            plt.show()


## Aim 1: Decoding And Specificity Checkpoints

This extraction scans available `cell_06*.joblib` and `*aim1*.joblib` payloads for scalar accuracy, permutation, cross-decoding, and similarity-like metrics. Payload schemas can vary, so extracted key names are shown directly.


In [ ]:
def flatten_payload(obj, prefix="", max_depth=4):
    rows = []
    if max_depth < 0:
        return rows
    if isinstance(obj, dict):
        for key, value in obj.items():
            name = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(value, (dict, list, tuple)) and max_depth > 0:
                rows.extend(flatten_payload(value, name, max_depth - 1))
            elif np.isscalar(value) or isinstance(value, (str, int, float, bool, np.number)):
                rows.append((name, value))
    elif isinstance(obj, (list, tuple)):
        for idx, value in enumerate(obj[:20]):
            name = f"{prefix}[{idx}]"
            if isinstance(value, (dict, list, tuple)) and max_depth > 0:
                rows.extend(flatten_payload(value, name, max_depth - 1))
            elif np.isscalar(value) or isinstance(value, (str, int, float, bool, np.number)):
                rows.append((name, value))
    return rows

def load_joblib(path):
    if joblib is None:
        return None
    try:
        return joblib.load(path)
    except Exception as exc:
        print(f"Could not load {path}: {exc}")
        return None

rows = []
for path in sorted(RESULT_ROOT.glob("**/cell_06*.joblib")) + sorted(RESULT_ROOT.glob("**/*aim1*.joblib")):
    payload = load_joblib(path)
    if payload is None:
        continue
    for key, value in flatten_payload(payload):
        lower = key.lower()
        if any(token in lower for token in ["accuracy", "acc", "p_value", "pval", "p_perm", "auc", "cosine", "similarity"]):
            rows.append({"source": str(path.relative_to(REPO_ROOT)), "metric_key": key, "value": value})
decoding_summary = pd.DataFrame(rows)
if decoding_summary.empty:
    display(Markdown("_No Aim 1 decoding checkpoint metrics were found under the configured result root._"))
else:
    decoding_summary["value_numeric"] = pd.to_numeric(decoding_summary["value"], errors="coerce")
    display_table(decoding_summary, n=80, caption="Extracted decoding/checkpoint scalar metrics")
    acc_like = decoding_summary.dropna(subset=["value_numeric"])
    acc_like = acc_like[acc_like["metric_key"].str.contains("acc|accuracy|auc|cosine|similarity", case=False, regex=True)]
    if not acc_like.empty:
        plt.figure(figsize=(10, max(4, 0.35 * len(acc_like))))
        sns.barplot(data=acc_like, y="metric_key", x="value_numeric", color="#5B7C99")
        plt.xlabel("Value")
        plt.ylabel("")
        plt.title("Aim 1 extracted decoding and specificity metrics")
        savefig("aim1_decoding_checkpoint_metrics.png")
        plt.show()


## Aim 2-5 Inferential Result Figures

The cells below use exported CSV tables when present. They produce forest plots for Aim 2 and Aim 5, heatmaps for Aim 3 and Aim 4, and a combined sensitivity estimate plot.


In [ ]:
def forest_plot(df, x="estimate", y="metric", hue=None, title="Forest plot", xlabel="Estimate", filename="forest.png"):
    if df.empty or x not in df.columns or y not in df.columns:
        display(Markdown(f"_Cannot draw `{title}` because required columns are missing._"))
        return
    plot_df = coerce_numeric(df.dropna(subset=[x]).copy(), [x, "ci_low", "ci_high"])
    if plot_df.empty:
        display(Markdown(f"_No numeric estimates for `{title}`._"))
        return
    plt.figure(figsize=(9, max(4, 0.45 * plot_df[y].nunique())))
    ax = sns.pointplot(data=plot_df, y=y, x=x, hue=hue, join=False, errorbar=None)
    order = [tick.get_text() for tick in ax.get_yticklabels()]
    for _, row in plot_df.iterrows():
        if pd.notna(row.get("ci_low")) and pd.notna(row.get("ci_high")) and row.get(y) in order:
            ypos = order.index(row[y])
            ax.plot([row["ci_low"], row["ci_high"]], [ypos, ypos], color="#333333", linewidth=1.1, alpha=0.75)
    ax.axvline(0, color="#333333", linewidth=1, linestyle="--")
    plt.xlabel(xlabel)
    plt.ylabel("")
    plt.title(title)
    if hue:
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    savefig(filename)
    plt.show()

def heatmap_plot(df, index, columns, values="estimate", title="Heatmap", filename="heatmap.png"):
    if df.empty or not {index, columns, values}.issubset(df.columns):
        display(Markdown(f"_Cannot draw `{title}` because required columns are missing._"))
        return
    matrix = df.pivot_table(index=index, columns=columns, values=values, aggfunc="first")
    if matrix.empty:
        display(Markdown(f"_No matrix values for `{title}`._"))
        return
    plt.figure(figsize=(max(7, 1.1 * matrix.shape[1]), max(3.5, 0.55 * matrix.shape[0])))
    sns.heatmap(matrix, center=0, cmap="vlag", annot=True, fmt=".2g", linewidths=0.5)
    plt.title(title)
    plt.xlabel(columns)
    plt.ylabel(index)
    savefig(filename)
    plt.show()


In [ ]:
aim2 = coerce_numeric(add_metric_role(stats_tables["Aim 2 group difference"]), ["estimate", "ci_low", "ci_high", "p", "q", "n"])
if aim2.empty:
    display(Markdown("_No `aim2_group_difference.csv` table is available yet._"))
else:
    aim2_view = aim2[aim2.get("metric_role", pd.Series(index=aim2.index, dtype=object)).eq("primary")].copy()
    if aim2_view.empty:
        aim2_view = aim2.copy()
    display_table(aim2_view.sort_values("p", na_position="last"), n=60, caption="Aim 2 SAD-HC group difference models")
    forest_plot(aim2_view, title="Aim 2 placebo SAD-HC group differences", xlabel="SAD minus HC estimate", filename="aim2_group_difference_forest.png")

aim3 = coerce_numeric(add_metric_role(stats_tables["Aim 3 clinical relevance"]), ["estimate", "ci_low", "ci_high", "p", "q", "n"])
if aim3.empty:
    display(Markdown("_No `aim3_clinical_relevance.csv` table is available yet._"))
else:
    score_col = "clinical_score" if "clinical_score" in aim3.columns else "clinical_score_z" if "clinical_score_z" in aim3.columns else None
    aim3_view = aim3.copy()
    if score_col:
        aim3_view = aim3_view[aim3_view[score_col].isin(PRIMARY_CLINICAL_SCORES)]
    if "metric_role" in aim3_view.columns:
        aim3_view = aim3_view[aim3_view["metric_role"].eq("primary")]
    if aim3_view.empty:
        aim3_view = aim3.copy()
    display_table(aim3_view.sort_values("p", na_position="last"), n=80, caption="Aim 3 clinical relevance models")
    if score_col and "Group" in aim3_view.columns:
        aim3_view["group_score"] = aim3_view["Group"].astype(str) + " | " + aim3_view[score_col].astype(str)
        heatmap_plot(aim3_view, index="metric", columns="group_score", title="Aim 3 clinical association estimates", filename="aim3_clinical_relevance_heatmap.png")


In [ ]:
aim4 = coerce_numeric(add_metric_role(stats_tables["Aim 4 SCR convergence"]), ["estimate", "ci_low", "ci_high", "p", "q", "n"])
if aim4.empty:
    display(Markdown("_No `aim4_scr_convergence.csv` table is available yet._"))
else:
    aim4_view = aim4.copy()
    if "scr_index" in aim4_view.columns:
        aim4_view = aim4_view[aim4_view["scr_index"].isin(PRIMARY_SCR_INDICES)]
    if "metric_role" in aim4_view.columns:
        aim4_view = aim4_view[aim4_view["metric_role"].eq("primary")]
    if aim4_view.empty:
        aim4_view = aim4.copy()
    display_table(aim4_view.sort_values("p", na_position="last"), n=100, caption="Aim 4 neural-SCR convergence models")
    if "Group" in aim4_view.columns:
        for group, sub in aim4_view.groupby("Group", dropna=False):
            heatmap_plot(sub, index="scr_index", columns="metric", title=f"Aim 4 neural-SCR convergence estimates: {group}", filename=f"aim4_scr_convergence_{clean_label(group)}.png")
    else:
        heatmap_plot(aim4_view, index="scr_index", columns="metric", title="Aim 4 neural-SCR convergence estimates", filename="aim4_scr_convergence.png")

aim5 = coerce_numeric(add_metric_role(stats_tables["Aim 5 oxytocin modulation"]), ["estimate", "ci_low", "ci_high", "p", "q", "n"])
if aim5.empty:
    display(Markdown("_No `aim5_oxytocin_modulation.csv` table is available yet._"))
else:
    aim5_view = aim5[aim5.get("metric_role", pd.Series(index=aim5.index, dtype=object)).eq("primary")].copy() if "metric_role" in aim5.columns else aim5.copy()
    if "term" in aim5_view.columns:
        gxdrug = aim5_view[aim5_view["term"].astype(str).str.contains("Group|Drug|:", case=False, regex=True)].copy()
        if not gxdrug.empty:
            aim5_view = gxdrug
    if aim5_view.empty:
        aim5_view = aim5.copy()
    display_table(aim5_view.sort_values("p", na_position="last"), n=80, caption="Aim 5 oxytocin modulation models")
    forest_plot(aim5_view, hue="term" if "term" in aim5_view.columns else None, title="Aim 5 oxytocin modulation effects", xlabel="Model estimate", filename="aim5_oxytocin_modulation_forest.png")


In [ ]:
sensitivity = coerce_numeric(add_metric_role(stats_tables["Sensitivity models"]), ["estimate", "ci_low", "ci_high", "p", "q", "n"])
if sensitivity.empty:
    display(Markdown("_No `sensitivity_models_all.csv` table is available yet._"))
else:
    display_table(sensitivity.sort_values("p", na_position="last"), n=100, caption="Sensitivity model table")
    if {"metric", "estimate"}.issubset(sensitivity.columns):
        label_cols = available_columns(sensitivity, ["analysis", "sensitivity", "feature_space"])
        plot_df = sensitivity.dropna(subset=["estimate"]).copy()
        plot_df["source"] = plot_df[label_cols].astype(str).agg(" | ".join, axis=1) if label_cols else "sensitivity"
        plt.figure(figsize=(10, max(5, 0.35 * min(len(plot_df), 80))))
        sns.scatterplot(data=plot_df.head(80), y="metric", x="estimate", hue="source", s=70)
        plt.axvline(0, color="#333333", linewidth=1, linestyle="--")
        plt.xlabel("Estimate")
        plt.ylabel("")
        plt.title("Sensitivity model estimates")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
        savefig("sensitivity_model_estimates.png")
        plt.show()

primary = coerce_numeric(stats_tables["Manuscript primary results"], ["estimate", "ci_low", "ci_high", "p", "q", "n"])
if primary.empty:
    display(Markdown("_No `manuscript_primary_results.csv` table is available yet._"))
else:
    display_table(primary, n=120, caption="Manuscript primary results")
    if {"aim", "estimate"}.issubset(primary.columns):
        label_cols = available_columns(primary, ["aim", "Group", "clinical_score_label", "scr_index", "metric"])
        plot_df = primary.dropna(subset=["estimate"]).copy()
        plot_df["row_label"] = plot_df[label_cols].fillna("").astype(str).agg(" | ".join, axis=1).str.replace(r"( \| )+", " | ", regex=True).str.strip(" |")
        plt.figure(figsize=(9, max(4, 0.35 * len(plot_df))))
        sns.barplot(data=plot_df, y="row_label", x="estimate", hue="aim", dodge=False)
        plt.axvline(0, color="#333333", linewidth=1, linestyle="--")
        plt.xlabel("Estimate")
        plt.ylabel("")
        plt.title("Manuscript primary effects")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
        savefig("manuscript_primary_effects.png")
        plt.show()


## Reproducibility Notes

Use this final cell as a lightweight readiness check before exporting figures into manuscript drafts.


In [ ]:
notes = []
notes.append(f"Result root: `{RESULT_ROOT}`")
notes.append(f"Stats directory exists: `{STATS_DIR.exists()}`")
notes.append(f"Subject metrics available: `{not subject_metrics.empty}`")
notes.append(f"SCR sensitivity groups available: `{not scr_groups.empty}`")
missing_tables = [label for label, table in stats_tables.items() if table.empty]
if missing_tables:
    notes.append("Missing or empty tables: " + ", ".join(f"`{label}`" for label in missing_tables))
else:
    notes.append("All expected stats tables were found.")
notes.append("Figures produced by this notebook are written to `" + str(FIGURE_DIR.relative_to(REPO_ROOT)) + "`.")
display(Markdown("\n".join(f"- {note}" for note in notes)))
